# PySpark 7: monitoring and performance

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark-07-monitoring-and-performance.ipynb)

A slow job is usually a plan problem: a skewed key, a shuffle that could have been a broadcast, or too many small files. This module shows how to read the plan and the Spark UI, then the settings worth changing.

**You will learn to:**

- Read a physical plan and find the expensive node
- Spot skew, spill and small-file problems in the Spark UI
- Size shuffle partitions and use adaptive query execution
- Decide when to cache and how to lay files out

**Before you start:** PySpark 1 to 6.

## Setup and sample data

This module runs on the same four tables as PySpark 1, so each module can run on its own. The setup cell installs PySpark when it is missing and starts a local session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## Sample data: the tables every lesson below runs on

This section seeds a small four-table slice of an orders platform, built with
`spark.createDataFrame` and an explicit schema per table. Three quirks the lessons lean on:
customer 4 has no `region`, customers 7 and 8 have never ordered, and one event is an exact
retry of the same row.

| table | grain | notable columns |
| --- | --- | --- |
| `customers` | one row per customer | `region`, `signup_date` (`DATE`) |
| `orders` | one row per order | `status`, `order_ts` (`TIMESTAMP`), `amount` (decimal) |
| `order_items` | one row per order line | `qty`, `unit_price` |
| `events` | one row per user event | `user_id`, `event`, `ts` |

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import types as T

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), False),
])

customers = spark.createDataFrame(
    [
        (1, "Acme Corp",      "North", date(2024, 1, 3)),
        (2, "Bright Retail",  "North", date(2024, 2, 11)),
        (3, "Cyan Traders",   "South", date(2024, 3, 15)),
        (4, "Delta Grocers",  None,    date(2024, 5, 2)),
        (5, "Echo Supplies",  "West",  date(2024, 6, 20)),
        (6, "Foxtrot Foods",  "West",  date(2024, 8, 1)),
        (7, "Gamma Sports",   "East",  date(2024, 9, 14)),
        (8, "Hotel Homeware", "East",  date(2024, 11, 5)),
    ],
    schema=customers_schema,
)

customers.orderBy("customer_id").show()

In [ ]:
from pyspark.sql import types as T

orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders = spark.createDataFrame(
    [
        (1,  1, "paid",     datetime(2026, 1,  5,  9, 15), D("120.00")),
        (2,  1, "shipped",  datetime(2026, 1,  5, 10,  5),  D("84.50")),
        (3,  1, "returned", datetime(2026, 1,  5, 11, 40),  D("30.00")),
        (4,  2, "paid",     datetime(2026, 1,  6, 15, 20), D("260.00")),
        (5,  2, "placed",   datetime(2026, 1,  7,  8,  5),  D("55.25")),
        (6,  3, "shipped",  datetime(2026, 1,  7, 17, 45), D("149.00")),
        (7,  3, "paid",     datetime(2026, 1,  9, 12, 10), D("310.00")),
        (8,  4, "returned", datetime(2026, 1, 10, 19, 30),  D("89.90")),
        (9,  4, "shipped",  datetime(2026, 1, 12,  8, 55), D("1180.00")),
        (10, 5, "paid",     datetime(2026, 1, 12, 13,  0),   D("74.00")),
        (11, 6, "placed",   datetime(2026, 1, 13, 10, 25),  D("210.00")),
        (12, 6, "shipped",  datetime(2026, 1, 14,  0, 30),   D("96.40")),
    ],
    schema=orders_schema,
)

orders.orderBy("order_id").show()
orders.printSchema()

In [ ]:
from pyspark.sql import types as T

order_items_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("sku", T.StringType(), False),
    T.StructField("qty", T.IntegerType(), False),
    T.StructField("unit_price", T.DecimalType(8, 2), False),
])

order_items = spark.createDataFrame(
    [
        (1,  "SKU-4001",  2,  D("12.50")),
        (1,  "SKU-4007",  1,  D("95.00")),
        (2,  "SKU-4002",  1,  D("34.50")),
        (2,  "SKU-4009",  2,  D("25.00")),
        (3,  "SKU-4002",  1,  D("30.00")),
        (4,  "SKU-4003",  1, D("149.00")),
        (4,  "SKU-4004",  2,  D("55.50")),
        (5,  "SKU-4005",  1,  D("55.25")),
        (6,  "SKU-4003",  1, D("149.00")),
        (7,  "SKU-4006",  1, D("189.00")),
        (7,  "SKU-4008",  1, D("121.00")),
        (8,  "SKU-4010",  1,  D("89.90")),
        (9,  "SKU-4011", 20,  D("59.00")),
        (10, "SKU-4012",  2,  D("37.00")),
        (11, "SKU-4013",  1, D("210.00")),
        (12, "SKU-4014",  1,  D("41.40")),
        (12, "SKU-4005",  1,  D("55.00")),
    ],
    schema=order_items_schema,
)

order_items.orderBy("order_id", "sku").show(20)

In [ ]:
from pyspark.sql import types as T

events_schema = T.StructType([
    T.StructField("user_id", T.IntegerType(), False),
    T.StructField("event", T.StringType(), False),
    T.StructField("ts", T.TimestampType(), False),
])

events = spark.createDataFrame(
    [
        (1, "view",        datetime(2026, 1,  6,  9,  0)),
        (1, "add_to_cart", datetime(2026, 1,  6,  9,  2)),
        (1, "purchase",    datetime(2026, 1,  6,  9, 10)),
        (2, "view",        datetime(2026, 1,  7, 10,  0)),
        (2, "purchase",    datetime(2026, 1,  7, 10, 40)),
        (3, "view",        datetime(2026, 1,  6, 11, 15)),
        (3, "view",        datetime(2026, 1,  8, 11, 20)),
        (4, "add_to_cart", datetime(2026, 1,  9, 20,  5)),
        (4, "purchase",    datetime(2026, 1, 10,  8, 30)),
        (5, "view",        datetime(2026, 1, 11, 14, 45)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
        (5, "purchase",    datetime(2026, 1, 12, 12, 30)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
    ],
    schema=events_schema,
)

events.orderBy("user_id", "ts").show()

In [ ]:
from pyspark.sql import functions as F

frames = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "events": events,
}
expected_rows = {"customers": 8, "orders": 12, "order_items": 17, "events": 13}

for table, frame in frames.items():
    actual = frame.count()
    assert actual == expected_rows[table], (table, actual, expected_rows[table])
    print(f"{table:<12} {actual:>2} rows")

per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines"))
    .join(orders.select("order_id", "amount"), "order_id")
)
assert per_order.where(F.col("lines") != F.col("amount")).count() == 0
print("line totals reconcile with the orders ledger:", float(orders.agg(F.sum("amount")).first()[0]))

assert events.dropDuplicates().count() == events.count() - 1
print("and the retried event collapses to 12 distinct rows - seed verified")

In [ ]:
import json
from pyspark.sql import Window, functions as F
from pyspark.sql.functions import broadcast

In [ ]:
daily_revenue = (
    orders
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)
print(type(daily_revenue).__name__, "- nothing has executed yet")

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()


orders_path = os.path.join(WORK_DIR, "orders_parquet")
orders.write.mode("overwrite").parquet(orders_path)

result = (
    spark.read.parquet(orders_path)
    .where("order_ts >= timestamp '2026-01-10'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)
plan = plan_of(result)
print(plan)
read_schema = next(line for line in plan.splitlines() if line.startswith("ReadSchema"))
assert "GreaterThan(amount,0.00)" in plan  # the filter reached the Parquet reader
assert "status" not in read_schema  # unused columns are never read

In [ ]:
from pyspark.sql import functions as F

enriched = (
    orders.alias("o")
    .join(customers.alias("c"), on="customer_id", how="left")
    .select(
        "o.order_id",
        "o.customer_id",
        "c.name",
        "c.region",
        "o.status",
        "o.amount",
    )
    .orderBy("o.order_id")
)
enriched.show()

# `customer_id` is unique on the dimension, so the enrich join must not fan out
assert enriched.count() == orders.count()
print("join verified: one row per order")

In [ ]:
bronze_orders_raw = spark.createDataFrame(
    [(1, 10, D("50.00")), (2, None, D("20.00")), (None, 11, D("5.00")), (4, 12, D("-3.00")), (5, 13, D("8.00"))],
    "order_id int, customer_id int, amount decimal(12,2)",
)

checked = bronze_orders_raw.withColumn(
    "quality_error",
    F.when(F.col("order_id").isNull(), "missing_order_id")
     .when(F.col("customer_id").isNull(), "missing_customer_id")
     .when(F.col("amount") < 0, "negative_amount"),
).cache()  # both writes below read the same checked frame

valid_orders = checked.where("quality_error is null").drop("quality_error")
quarantine = checked.where("quality_error is not null").withColumn("rejected_at", F.current_timestamp())

valid_orders.write.mode("overwrite").saveAsTable("silver_orders_checked")
quarantine.write.mode("overwrite").saveAsTable("ops_orders_quarantine")

quarantine.groupBy("quality_error").count().orderBy("quality_error").show()
assert valid_orders.count() == 2 and quarantine.count() == 3
checked.unpersist()

## How do you monitor and debug PySpark jobs?

Start with the Spark UI. It shows jobs, stages, tasks, shuffle read/write, spilled memory, input size, executor failures, and SQL plans.

Useful PySpark checks:

In [ ]:
print(plan_of(enriched, "simple"))
print("partitions:", enriched.rdd.getNumPartitions())
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Common symptoms:

+ One task runs much longer than others: likely skew
+ Huge shuffle read/write: expensive join, aggregation, or repartition
+ Spilled memory: insufficient executor memory or too-wide shuffle
+ Many tiny tasks: small files or too many partitions
+ Driver out of memory: collecting too much data to the driver

For production jobs, log record counts at each pipeline boundary:

In [ ]:
input_count = bronze_orders_raw.count()
output_count = valid_orders.count()

spark.createDataFrame(
    [("orders_silver", input_count, output_count)],
    "pipeline_name string, input_count long, output_count long",
).write.mode("append").saveAsTable("ops_pipeline_metrics")

spark.table("ops_pipeline_metrics").show()
assert (input_count, output_count) == (5, 2)

How it works under the hood. The driver's DAG scheduler tracks stages and tasks and emits the event log the Spark UI reads. The numbers that matter are per-stage: shuffle read and write bytes, spill to memory and disk, the task duration distribution, and GC time. A stage boundary exists wherever the plan contains an `Exchange`.

Worked example. If 199 tasks in a stage finish in 20 seconds and one takes 600 seconds, the stage wall time is 600 seconds. Adding partitions will not help; the hot key must be split with AQE skew join or manual salting.

Common mistake: looking only at the average task time. The max-to-median ratio is the skew signal, and excessive driver-side `collect()` shows up as driver GC, not as a slow task.

## How do you make a PySpark job faster?

Tuning starts with the plan and the Spark UI, not with a config change. Find the widest stage, read its shuffle bytes, spill and task-duration spread, then fix the one thing that dominates. The levers below are ordered by how often they matter. The deeper recipes live in [Spark Tuning](spark-tuning.ipynb).

**Adaptive Query Execution (AQE).** AQE re-optimizes the plan after shuffle stages report real sizes. It coalesces many small post-shuffle partitions, splits oversized partitions for skewed joins, and can switch a sort-merge join to a broadcast join at runtime. It is on by default since Spark 3.2.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "64m")
print({key: spark.conf.get(key) for key in ("spark.sql.adaptive.enabled", "spark.sql.adaptive.skewJoin.enabled")})

**Shuffle partition sizing.** `spark.sql.shuffle.partitions` sets the reduce fan-out for every shuffle; the default 200 is wrong for both tiny and huge jobs. Aim for partitions of roughly 128-256 MB after the shuffle, and let AQE coalesce the small ones instead of hand-tuning per query.

In [ ]:
shuffle_bytes = 200 * 1024 ** 3
target_partition_bytes = 128 * 1024 ** 2
partitions = -(-shuffle_bytes // target_partition_bytes)  # ceiling division
print("target partitions:", partitions)

**Broadcast versus sort-merge join.** A broadcast join ships the small side to every executor and avoids shuffling the large side; it is chosen when the estimated side is under `spark.sql.autoBroadcastJoinThreshold` (10 MB by default). Fall back to sort-merge when neither side is small, and check the physical plan to confirm what was chosen.

In [ ]:
threshold_bytes = int(spark.conf.get("spark.sql.autoBroadcastJoinThreshold").rstrip("b"))
print("broadcast threshold bytes:", threshold_bytes)

enriched = orders.join(broadcast(customers), on="customer_id", how="left")
assert "BroadcastHashJoin" in plan_of(enriched)
assert threshold_bytes == 10 * 1024 * 1024

**Skew.** One hot key puts most rows in one partition, so one task defines the stage duration. AQE skew join splits it when the partition exceeds both `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (256 MB) and 5x the median size. Otherwise, salt the hot key: add a random bucket to the fact side and explode the dimension across the same buckets.

In [ ]:
salt_buckets = 16
salted_facts = orders.withColumn("salt", (F.rand(seed=42) * salt_buckets).cast("int"))
salted_dim = customers.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(salt_buckets)])))
joined = salted_facts.join(salted_dim, ["customer_id", "salt"]).drop("salt")

# salting changes how rows are spread, never which rows match
assert joined.count() == orders.join(customers, "customer_id").count()
print("salted join rows:", joined.count())

**Caching.** Cache only a frame that several actions reuse or that feeds an iterative loop. Caching is lazy: the first action materialises the blocks, and you should `unpersist()` when done. A cached frame used once just adds memory pressure.

In [ ]:
enriched = orders.join(broadcast(customers), "customer_id").cache()
enriched.count()  # the first action materializes the cached blocks
enriched.groupBy("region").count().orderBy("region").show()
assert enriched.is_cached
enriched.unpersist()

**File layout and predicate pushdown.** Partition on a low-cardinality column that appears in filters, target 100 MB-1 GB per file, and write a sensible number of files with `repartition` or `coalesce`. On the read side, `spark.sql.files.maxPartitionBytes` (128 MB) controls how much one task reads, and Parquet or Delta statistics skip files and row groups that cannot match the predicate.

In [ ]:
curated_path = os.path.join(WORK_DIR, "curated_orders")
(
    orders
    .withColumn("order_date", F.to_date("order_ts"))
    .repartition("order_date")
    .write.mode("overwrite")
    .partitionBy("order_date")
    .parquet(curated_path)
)

pruned = spark.read.parquet(curated_path).filter("order_date = '2026-01-12'")
plan = plan_of(pruned)
print(plan)
assert "PartitionFilters: [isnotnull(order_date" in plan
assert sorted(r["order_id"] for r in pruned.collect()) == [9, 10]

That is the short list. For the full treatment of AQE, partition sizing, join strategies, skew, storage layout and profiling, see [Spark Tuning](spark-tuning.ipynb).

## Check yourself

The lessons above end with takeaways. Re-run each cell in order, and use `explain` on any query you do not understand before you move on.

## Key takeaways

- Fix the plan before you change configuration.
- One slow task out of 200 is skew, not a slow cluster. Salt the key or isolate the hot value.
- Cache only a DataFrame that several actions reuse.

**Next:** [PySpark 8: real-world use cases and interviews](pyspark-08-real-world-and-interviews.ipynb)